# Counterfactual Dataset Generation with Gemini API

**Pipeline**
1. Stratified random sampling from source datasets (outside the LLM)
2. Generation phase — send batches to Gemini with the appropriate system prompt
3. Audit phase — send all `action=ACCEPT` items to the auditor prompt
4. Filter — keep only pairs where **both** `action=ACCEPT` and `verdict=ACCEPT`
5. Export — final JSONL (rich metadata) + CSV (flat, with grouped train/test split)

**Sources → prompts**
| Dataset | Direction | Generation prompt | Audit prompt |
|---|---|---|---|
| `biased_corpus_only.json` | biased → neutral | Prompt 1 | Prompt 3 |
| `gemini_only.json` | neutral → biased | Prompt 2 | Prompt 4 |
| `gus_only.json` | biased → neutral | Prompt 1 | Prompt 3 | (re-run with `BIASED_CORPUS_PATH = gus_only.json`)

> **Note.** The notebook was run a second time on `gus_only.json` with `BIASED_CORPUS_PATH` re-pointed at the GUS source. The same Prompt 1 and Prompt 3 are used; only the biased→neutral direction is applied to GUS, since GUS already contributes the biased side of the GUS pairs.

All intermediate results are checkpointed to JSONL files inside `dataset/v2/output/`.  
Re-running any cell resumes from the last checkpoint automatically.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
# Path to the "Dataset Bias" folder
dataset_bias_path = '/content/drive/MyDrive/Dataset Bias'
# Check that the folder exists
if os.path.exists(dataset_bias_path):
    print("Found the 'Dataset Bias' folder.")
else:
    print("The 'Dataset Bias' folder was not found.")

Found the 'Dataset Bias' folder.


In [ ]:
%pip install google-generativeai pandas scikit-learn tqdm -q


In [ ]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from tqdm.notebook import tqdm

import google.generativeai as genai


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Configuration
Set your API key and adjust sampling/batching parameters before running.

In [ ]:
# ── API key ──────────────────────────────────────────────────────────────────
# Either set the GOOGLE_API_KEY environment variable or paste your key below.
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
genai.configure(api_key=GOOGLE_API_KEY)

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "gemini-3.1-pro-preview"  # or "gemini-1.5-flash", "gemini-2.0-flash-lite"

# ── Sampling ──────────────────────────────────────────────────────────────────
SAMPLE_SIZE_BIASED  = 2000   # entries to draw from biased_corpus_only.json
SAMPLE_SIZE_NEUTRAL = 2317   # entries to draw from gemini_only.json
RANDOM_SEED = 42

# ── API call parameters ───────────────────────────────────────────────────────
BATCH_SIZE        = 20    # items per API call (lower if hitting token limits)
RATE_LIMIT_DELAY  = 1.5   # seconds between calls
MAX_RETRIES       = 3

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR = Path("/content/drive/MyDrive/Dataset Bias")
V2_DIR   = BASE_DIR
OUT_DIR  = V2_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BIASED_CORPUS_PATH  = BASE_DIR / "biased_corpus_only.json"
NEUTRAL_CORPUS_PATH = BASE_DIR / "gemini_only.json"

PROMPT_1_PATH = V2_DIR / "1-Prompt biased-neutral.txt"
PROMPT_2_PATH = V2_DIR / "2-Prompt neutral-biased.txt"
PROMPT_3_PATH = V2_DIR / "3-Prompt Auditoria biased-neutral.txt"
PROMPT_4_PATH = V2_DIR / "4-Prompt Auditoria neutral-biased.txt"

GEN_A_PATH   = OUT_DIR / "generation_biased_neutral.jsonl"
GEN_B_PATH   = OUT_DIR / "generation_neutral_biased.jsonl"
AUDIT_A_PATH = OUT_DIR / "audit_biased_neutral.jsonl"
AUDIT_B_PATH = OUT_DIR / "audit_neutral_biased.jsonl"
FINAL_JSONL  = OUT_DIR / "final_counterfactuals.jsonl"
FINAL_CSV    = OUT_DIR / "final_counterfactuals.csv"

# ── Sanity checks ─────────────────────────────────────────────────────────────
if GOOGLE_API_KEY == "YOUR_API_KEY_HERE":
    print("WARNING: GOOGLE_API_KEY not set – API calls will fail.")
else:
    print("API key configured.")
print("Output directory:", OUT_DIR.resolve())


API key configured.
Output directory: /content/drive/MyDrive/Dataset Bias/output


## Helper functions

In [ ]:
# ── Prompt loader ─────────────────────────────────────────────────────────────

def load_prompt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()


# ── JSONL response parser ─────────────────────────────────────────────────────

def parse_jsonl_response(text):
    """Parse JSONL output from Gemini, tolerating markdown code fences."""
    lines = text.strip().splitlines()
    cleaned = []
    for line in lines:
        s = line.strip()
        if not s or s.startswith("```"):
            continue
        cleaned.append(s)

    results, errors = [], []
    for i, line in enumerate(cleaned):
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError as exc:
            errors.append({"line": i, "preview": line[:120], "error": str(exc)})

    if errors:
        print(f"  WARNING: {len(errors)} JSON parse error(s) in response")
        for e in errors[:3]:
            print(f"    line {e['line']}: {e['preview']}")
    return results


# ── Gemini call with retry ─────────────────────────────────────────────────────

def call_gemini(system_prompt, user_content):
    """Call Gemini with exponential-backoff retry. Returns response text or None."""
    model = genai.GenerativeModel(
        MODEL_NAME,
        system_instruction=system_prompt,
    )
    gen_cfg = genai.types.GenerationConfig(
        temperature=0.3,
        max_output_tokens=8192,
    )
    for attempt in range(MAX_RETRIES):
        try:
            response = model.generate_content(user_content, generation_config=gen_cfg)
            return response.text
        except Exception as exc:
            if attempt < MAX_RETRIES - 1:
                delay = 2.0 * (2 ** attempt)
                print(f"  Attempt {attempt + 1} failed ({exc}). Retrying in {delay:.0f}s...")
                time.sleep(delay)
            else:
                print(f"  All {MAX_RETRIES} attempts failed: {exc}")
                return None


# ── JSONL checkpoint helpers ───────────────────────────────────────────────────

def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    results = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                try:
                    results.append(json.loads(s))
                except json.JSONDecodeError:
                    pass
    return results


def append_jsonl(path, items):
    with open(path, "a", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


def fmt_jsonl(items):
    """Serialise a list of dicts to a JSONL string (for use as LLM input)."""
    return "\n".join(json.dumps(item, ensure_ascii=False) for item in items)


print("Helpers loaded.")


Helpers loaded.


## 1. Load datasets

In [ ]:
with open(BIASED_CORPUS_PATH, "r", encoding="utf-8") as f:
    biased_data = json.load(f)["entries"]

with open(NEUTRAL_CORPUS_PATH, "r", encoding="utf-8") as f:
    neutral_data = json.load(f)["entries"]

print(f"biased_corpus_only : {len(biased_data)} entries")
print(f"gemini_only        : {len(neutral_data)} entries")

print("\nSample biased entry:")
print(json.dumps(biased_data[0], indent=2, ensure_ascii=False))

print("\nSample neutral entry:")
print(json.dumps(neutral_data[0], indent=2, ensure_ascii=False))

# Stratification variable distribution
biased_df = pd.DataFrame(biased_data)
print("\nbias_type distribution (biased_corpus_only):")
print(biased_df["bias_type"].value_counts().to_string())


biased_corpus_only : 2476 entries
gemini_only        : 2317 entries

Sample biased entry:
{
  "id": 2,
  "type": "single",
  "text": "Ukrainian refugees are innovative problem solvers.",
  "has_bias": true,
  "bias_type": "nationality",
  "bias_description": "Discriminatory language targeting refugees"
}

Sample neutral entry:
{
  "id": 1,
  "type": "single",
  "text": "The consultant maintained high standards.",
  "has_bias": false,
  "bias_type": null,
  "bias_description": null,
  "topic": "everyday"
}

bias_type distribution (biased_corpus_only):
bias_type
gender           388
socioeconomic    299
racial           283
age              262
nationality      251
political        231
religious        176
disability       174
sexuality        158
educational      150
physical         104


In [ ]:
# ── Classify topic for gemini_only entries ────────────────────────────────────
# Neutral sentences don't have a bias_type, but we want thematic diversity
# in the sample. This cell classifies each sentence by domain/topic.

VALID_TOPICS = [
    "science", "technology", "health", "education", "politics",
    "economy", "environment", "culture", "sports", "law",
    "everyday", "other"
]

CLASSIFY_TOPIC_PROMPT = f"""You are a topic classifier for a research dataset of neutral sentences.

For each sentence, decide which **thematic domain** it belongs to.
Choose exactly ONE label from:

{json.dumps(VALID_TOPICS)}

── Input format (JSONL) ──
Each line: {{"id": <int>, "text": "<sentence>"}}

── Output format (JSONL — one line per input, same order) ──
{{"id": <int>, "topic": "<label>"}}

Rules:
- Output ONLY valid JSONL, no markdown fences, no extra text.
- Use exactly the label strings above (lowercase).
- Every input id must appear in the output.
"""

TOPIC_CHECKPOINT = OUT_DIR / "topic_classifications.jsonl"

# ── Gather neutral entries that need classification ───────────────────────────
to_classify = [{"id": e["id"], "text": e["text"]} for e in neutral_data]
print(f"gemini_only entries to classify: {len(to_classify)}")

# ── Load checkpoint ───────────────────────────────────────────────────────────
existing_cls = load_jsonl(TOPIC_CHECKPOINT)
classified_ids = {item["id"] for item in existing_cls}
remaining = [x for x in to_classify if x["id"] not in classified_ids]
print(f"Already classified: {len(existing_cls)} | Remaining: {len(remaining)}")

# ── Classify in batches ──────────────────────────────────────────────────────
if remaining:
    batches = [remaining[i:i + BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]

    for batch in tqdm(batches, desc="Classifying topic"):
        response_text = call_gemini(CLASSIFY_TOPIC_PROMPT, fmt_jsonl(batch))

        if response_text is None:
            print("  Skipping batch (API failure)")
            time.sleep(RATE_LIMIT_DELAY)
            continue

        parsed = parse_jsonl_response(response_text)

        for item in parsed:
            t = str(item.get("topic", "other")).strip().lower()
            if t not in VALID_TOPICS:
                t = "other"
            item["topic"] = t

        append_jsonl(TOPIC_CHECKPOINT, parsed)
        existing_cls.extend(parsed)
        time.sleep(RATE_LIMIT_DELAY)

print(f"\nTotal classifications: {len(existing_cls)}")

# ── Apply topic back to in-memory neutral_data ────────────────────────────────
topic_map = {item["id"]: item["topic"] for item in existing_cls}

updated = 0
for entry in neutral_data:
    if entry["id"] in topic_map:
        entry["topic"] = topic_map[entry["id"]]
        updated += 1

print(f"Updated in memory: {updated} / {len(neutral_data)} entries")

# ── Save updated gemini_only.json ─────────────────────────────────────────────
with open(NEUTRAL_CORPUS_PATH, "w", encoding="utf-8") as f:
    json.dump({"entries": neutral_data}, f, ensure_ascii=False, indent=4)
print(f"Saved {NEUTRAL_CORPUS_PATH}")

# ── Show distribution ─────────────────────────────────────────────────────────
print("\ntopic distribution (gemini_only):")
print(pd.DataFrame(neutral_data)["topic"].value_counts().to_string())

gemini_only entries to classify: 2317
Already classified: 2317 | Remaining: 0

Total classifications: 2317
Updated in memory: 2317 / 2317 entries
Saved /content/drive/MyDrive/Dataset Bias/gemini_only.json

topic distribution (gemini_only):
topic
everyday       691
culture        251
economy        206
technology     200
politics       182
health         180
science        159
law            154
education      146
environment     63
other           45
sports          40


## 2. Stratified random sampling

- `biased_corpus_only` → stratify by `bias_type`
- `gemini_only` → `bias_type` is null for all neutral sentences, so simple random sampling

Sampling is done **outside the LLM**, before any API call.

In [ ]:
def stratified_sample(entries, strat_col, n_sample, seed=42):
    """
    Proportionally stratified sample by strat_col.
    Falls back to simple random sampling if strat_col is absent or all-null.
    Each stratum gets at least 1 slot (if available).
    """
    df = pd.DataFrame(entries)
    n_sample = min(n_sample, len(df))

    if strat_col not in df.columns or df[strat_col].isna().all():
        print(f"  No usable '{strat_col}' column — simple random sample")
        return df.sample(n_sample, random_state=seed).to_dict("records")

    df = df.copy()
    df["_strat"] = df[strat_col].fillna("unknown")
    counts = df["_strat"].value_counts()
    n_total = len(df)

    sampled_dfs = []
    for stratum, count in counts.items():
        alloc = max(1, round(n_sample * count / n_total))
        alloc = min(alloc, count)
        sampled_dfs.append(
            df[df["_strat"] == stratum].sample(alloc, random_state=seed)
        )

    result = (
        pd.concat(sampled_dfs)
        .sample(frac=1, random_state=seed)  # shuffle
        .head(n_sample)
        .drop(columns=["_strat"])
    )

    print(f"  Sampled {len(result)} / {n_total} entries")
    print(f"  Distribution by '{strat_col}':")
    print(result[strat_col].value_counts().to_string())
    return result.to_dict("records")


print("=" * 55)
print("Sampling biased_corpus_only")
print("=" * 55)
sample_biased = stratified_sample(
    biased_data, "bias_type", SAMPLE_SIZE_BIASED, seed=RANDOM_SEED
)

print()
print("=" * 55)
print("Sampling gemini_only")
print("=" * 55)
sample_neutral = stratified_sample(
    neutral_data, "bias_type", SAMPLE_SIZE_NEUTRAL, seed=RANDOM_SEED
)

print(f"\nFinal sample sizes — biased: {len(sample_biased)}, neutral: {len(sample_neutral)}")


Sampling biased_corpus_only
  Sampled 2000 / 2476 entries
  Distribution by 'bias_type':
bias_type
gender           313
socioeconomic    242
racial           229
age              212
nationality      202
political        187
religious        141
disability       141
sexuality        128
educational      121
physical          84

Sampling gemini_only
  No usable 'bias_type' column — simple random sample

Final sample sizes — biased: 2000, neutral: 2317


## 3. Generation phase

Results are written to JSONL checkpoints. Rerunning a cell resumes from the last checkpoint — already-processed `id`s are skipped.

- **Phase A** — `biased_corpus_only` → neutral counterfactuals (Prompt 1)
- **Phase B** — `gemini_only` → biased counterfactuals (Prompt 2)

In [ ]:
def run_generation(samples, prompt_path, source_name, hasbias_value, output_path):
    """
    Run the generation phase for one direction.

    Parameters
    ----------
    samples       : list of dicts from the source dataset
    prompt_path   : Path to the system prompt .txt file
    source_name   : string label for the source (stored in output)
    hasbias_value : bool — hasbias of the ORIGINAL sentences
    output_path   : JSONL checkpoint file path

    Returns
    -------
    list of all result dicts (existing + newly generated)
    """
    output_path = Path(output_path)
    existing = load_jsonl(output_path)
    processed_ids = {str(item["id"]) for item in existing}

    remaining = [s for s in samples if str(s["id"]) not in processed_ids]
    print(f"[{source_name}] {len(existing)} already done | {len(remaining)} to process")

    if not remaining:
        print("  Checkpoint complete — nothing new to do.")
        _print_gen_summary(existing, source_name)
        return existing

    system_prompt = load_prompt(prompt_path)
    batches = [remaining[i:i + BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]
    new_items = []

    for batch in tqdm(batches, desc=f"Generating [{source_name}]"):
        # Map dataset fields to the schema the prompt expects
        batch_fmt = [
            {
                "id":      item["id"],
                "source":  source_name,
                "type":    item.get("type", "single"),
                "text":    item["text"],
                "hasbias": hasbias_value,
                "topic":   item.get("bias_type") or "general",
            }
            for item in batch
        ]

        response_text = call_gemini(system_prompt, fmt_jsonl(batch_fmt))
        if response_text is None:
            print(f"  Skipping batch (API failure)")
            time.sleep(RATE_LIMIT_DELAY)
            continue

        parsed = parse_jsonl_response(response_text)

        # Annotate each result with source metadata from the original item
        orig_map = {str(item["id"]): item for item in batch}
        for result_item in parsed:
            orig = orig_map.get(str(result_item.get("id", "")))
            if orig:
                result_item["source"]           = source_name
                result_item["hasbias_original"] = hasbias_value
                result_item["topic"]            = orig.get("bias_type") or "general"

        append_jsonl(output_path, parsed)
        new_items.extend(parsed)
        time.sleep(RATE_LIMIT_DELAY)

    all_results = existing + new_items
    _print_gen_summary(all_results, source_name)
    return all_results


def _print_gen_summary(results, label):
    accept = sum(1 for x in results if x.get("action") == "ACCEPT")
    skip   = sum(1 for x in results if x.get("action") == "SKIP")
    total  = len(results)
    rate   = accept / total * 100 if total else 0
    print(f"  [{label}] total={total} | ACCEPT={accept} ({rate:.1f}%) | SKIP={skip}")


print("Generation function defined.")


Generation function defined.


In [ ]:
# Phase A: biased_corpus_only → neutral counterfactuals
print("Phase A: biased → neutral")
gen_a_results = run_generation(
    sample_biased,
    prompt_path=PROMPT_1_PATH,
    source_name="biased_corpus_only",
    hasbias_value=True,
    output_path=GEN_A_PATH,
)


Phase A: biased → neutral
[biased_corpus_only] 2001 already done | 0 to process
  Checkpoint complete — nothing new to do.
  [biased_corpus_only] total=2001 | ACCEPT=1406 (70.3%) | SKIP=595


In [ ]:
# Phase B: gemini_only → biased counterfactuals
print("Phase B: neutral → biased")
gen_b_results = run_generation(
    sample_neutral,
    prompt_path=PROMPT_2_PATH,
    source_name="gemini_only",
    hasbias_value=False,
    output_path=GEN_B_PATH,
)


Phase B: neutral → biased
[gemini_only] 2317 already done | 0 to process
  Checkpoint complete — nothing new to do.
  [gemini_only] total=2317 | ACCEPT=1058 (45.7%) | SKIP=1259


## 4. Audit phase

Only items with `action=ACCEPT` from generation are sent to the auditor.

- **Audit A** — verify neutral counterfactuals (Prompt 3)
- **Audit B** — verify biased counterfactuals (Prompt 4)

In [ ]:
def run_audit(generation_results, prompt_path, intended_label, source_name, output_path):
    """
    Audit all ACCEPT items from generation_results.

    Parameters
    ----------
    generation_results : output of run_generation()
    prompt_path        : Path to the auditor system prompt .txt file
    intended_label     : "neutral" or "biased"
    source_name        : label for logging
    output_path        : JSONL checkpoint file path

    Returns
    -------
    list of all audit result dicts
    """
    output_path = Path(output_path)
    accept_items = [x for x in generation_results if x.get("action") == "ACCEPT"]
    print(f"[{source_name}] {len(accept_items)} ACCEPT items to audit "
          f"(of {len(generation_results)} total)")

    existing = load_jsonl(output_path)
    processed_pair_ids = {item["pair_id"] for item in existing if "pair_id" in item}
    remaining = [x for x in accept_items if x.get("pair_id") not in processed_pair_ids]
    print(f"  {len(existing)} already audited | {len(remaining)} remaining")

    if not remaining:
        print("  Checkpoint complete — nothing new to do.")
        _print_audit_summary(existing, source_name)
        return existing

    system_prompt = load_prompt(prompt_path)
    batches = [remaining[i:i + BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]
    new_items = []

    for batch in tqdm(batches, desc=f"Auditing [{source_name}]"):
        batch_fmt = [
            {
                "id":                 item["id"],
                "pair_id":            item["pair_id"],
                "source":             item.get("source", ""),
                "topic":              item.get("topic", "general"),
                "original_text":      item.get("original_text", ""),
                "counterfactual_text": item.get("counterfactual_text", ""),
                "intended_label":     intended_label,
            }
            for item in batch
        ]

        response_text = call_gemini(system_prompt, fmt_jsonl(batch_fmt))
        if response_text is None:
            print(f"  Skipping batch (API failure)")
            time.sleep(RATE_LIMIT_DELAY)
            continue

        parsed = parse_jsonl_response(response_text)
        append_jsonl(output_path, parsed)
        new_items.extend(parsed)
        time.sleep(RATE_LIMIT_DELAY)

    all_results = existing + new_items
    _print_audit_summary(all_results, source_name)
    return all_results


def _print_audit_summary(results, label):
    accept = sum(1 for x in results if x.get("verdict") == "ACCEPT")
    reject = sum(1 for x in results if x.get("verdict") == "REJECT")
    total  = len(results)
    rate   = accept / total * 100 if total else 0
    print(f"  [{label}] audited={total} | ACCEPT={accept} ({rate:.1f}%) | REJECT={reject}")


print("Audit function defined.")


Audit function defined.


In [ ]:
# Audit A: verify neutral counterfactuals
print("Audit A: biased → neutral")
audit_a_results = run_audit(
    gen_a_results,
    prompt_path=PROMPT_3_PATH,
    intended_label="neutral",
    source_name="biased_corpus_only",
    output_path=AUDIT_A_PATH,
)


Audit A: biased → neutral
[biased_corpus_only] 1406 ACCEPT items to audit (of 2001 total)
  715 already audited | 691 remaining


Auditing [biased_corpus_only]:   0%|          | 0/35 [00:00<?, ?it/s]

    line 7: {"id": 3774, "pair_id": "cf_neutral_3774", "verdict": "REJECT", "failure_reasons": ["meaning drift is too large", "minim
    line 14: {"id": 607, "pair_id": "cf_neutral_607", "verdict": "REJECT", "failure_reasons": ["meaning drift is too large", "minimal
    line 5: {"id": 5020, "pair_id": "cf_neutral_5020", "verdict": "REJECT", "failure_reasons": ["same_sentence_type": false], "bias_
    line 6: {"id": 4045, "pair_id": "cf_neutral_4045", "verdict": "REJECT", "failure_reasons": ["same_topic": false], "bias_present_
  [biased_corpus_only] audited=1402 | ACCEPT=1131 (80.7%) | REJECT=271


In [ ]:
# Audit B: verify biased counterfactuals
print("Audit B: neutral → biased")
audit_b_results = run_audit(
    gen_b_results,
    prompt_path=PROMPT_4_PATH,
    intended_label="biased",
    source_name="gemini_only",
    output_path=AUDIT_B_PATH,
)


Audit B: neutral → biased
[gemini_only] 1058 ACCEPT items to audit (of 2317 total)
  906 already audited | 153 remaining


Auditing [gemini_only]:   0%|          | 0/8 [00:00<?, ?it/s]

  [gemini_only] audited=1059 | ACCEPT=697 (65.8%) | REJECT=362


## 5. Filter — keep only double-ACCEPT pairs

A pair enters the final dataset only if:
- generation returned `action = ACCEPT`
- audit returned `verdict = ACCEPT`

Items with `SKIP` or `REJECT` are preserved in the checkpoint files for methodology documentation.

In [ ]:
def merge_and_filter(gen_results, audit_results, label):
    """Join on pair_id and keep only action=ACCEPT AND verdict=ACCEPT pairs."""
    audit_map = {item["pair_id"]: item for item in audit_results if "pair_id" in item}

    accepted, skipped, rejected, not_audited = [], [], [], []

    for gen_item in gen_results:
        pair_id = gen_item.get("pair_id", "")
        action  = gen_item.get("action", "")

        if action != "ACCEPT":
            skipped.append(gen_item)
            continue

        audit_item = audit_map.get(pair_id)
        if audit_item is None:
            not_audited.append(gen_item)
            continue

        if audit_item.get("verdict") == "ACCEPT":
            accepted.append({"generation": gen_item, "audit": audit_item})
        else:
            rejected.append({"generation": gen_item, "audit": audit_item})

    print(f"[{label}]")
    print(f"  Total generated : {len(gen_results)}")
    print(f"  Gen SKIP        : {len(skipped)}")
    print(f"  Gen ACCEPT      : {len(gen_results) - len(skipped)}")
    print(f"  Audit REJECT    : {len(rejected)}")
    print(f"  Not yet audited : {len(not_audited)}")
    print(f"  Final ACCEPT    : {len(accepted)}")
    if not_audited:
        print(f"  WARNING: {len(not_audited)} items were accepted in generation but have no audit result")
    return accepted


pairs_a = merge_and_filter(gen_a_results, audit_a_results, "Phase A: biased→neutral")
print()
pairs_b = merge_and_filter(gen_b_results, audit_b_results, "Phase B: neutral→biased")

all_pairs = pairs_a + pairs_b
print(f"\nTotal accepted pairs: {len(all_pairs)} ({len(pairs_a)} A + {len(pairs_b)} B)")


[Phase A: biased→neutral]
  Total generated : 2001
  Gen SKIP        : 595
  Gen ACCEPT      : 1406
  Audit REJECT    : 271
  Not yet audited : 4
  Final ACCEPT    : 1131

[Phase B: neutral→biased]
  Total generated : 2317
  Gen SKIP        : 1259
  Gen ACCEPT      : 1058
  Audit REJECT    : 362
  Not yet audited : 0
  Final ACCEPT    : 696

Total accepted pairs: 1827 (1131 A + 696 B)


## 6. Acceptance rate statistics

Useful numbers to report in the methodology section.

In [ ]:
def acceptance_stats(gen_results, audit_results, label):
    total_gen   = len(gen_results)
    gen_accept  = sum(1 for x in gen_results if x.get("action") == "ACCEPT")
    gen_skip    = sum(1 for x in gen_results if x.get("action") == "SKIP")

    total_audit  = len(audit_results)
    audit_accept = sum(1 for x in audit_results if x.get("verdict") == "ACCEPT")
    audit_reject = sum(1 for x in audit_results if x.get("verdict") == "REJECT")

    gen_rate    = gen_accept / total_gen * 100 if total_gen else 0
    audit_rate  = audit_accept / total_audit * 100 if total_audit else 0
    overall_rate = gen_accept * audit_accept / (total_gen * total_audit) * 100 \
                   if total_gen and total_audit else 0

    print(f"{'─'*55}")
    print(f"{label}")
    print(f"{'─'*55}")
    print(f"  Generation : {total_gen:4d} total | ACCEPT {gen_accept:4d} ({gen_rate:.1f}%) | SKIP {gen_skip:4d}")
    print(f"  Audit      : {total_audit:4d} total | ACCEPT {audit_accept:4d} ({audit_rate:.1f}%) | REJECT {audit_reject:4d}")
    print(f"  Overall pipeline acceptance rate : {overall_rate:.1f}%")
    print()


acceptance_stats(gen_a_results, audit_a_results, "Phase A: biased → neutral")
acceptance_stats(gen_b_results, audit_b_results, "Phase B: neutral → biased")

# Skip reasons
print("Skip reasons — Phase A:")
skip_a = [x.get("skip_reason", "") for x in gen_a_results if x.get("action") == "SKIP"]
for reason in sorted(set(skip_a)):
    print(f"  {skip_a.count(reason):3d}x  {reason[:80]}")

print()
print("Skip reasons — Phase B:")
skip_b = [x.get("skip_reason", "") for x in gen_b_results if x.get("action") == "SKIP"]
for reason in sorted(set(skip_b)):
    print(f"  {skip_b.count(reason):3d}x  {reason[:80]}")

# Reject failure reasons
print()
print("Audit reject reasons — Phase A:")
for item in audit_a_results:
    if item.get("verdict") == "REJECT":
        for r in item.get("failure_reasons", []):
            print(f"  - {r}")

print()
print("Audit reject reasons — Phase B:")
for item in audit_b_results:
    if item.get("verdict") == "REJECT":
        for r in item.get("failure_reasons", []):
            print(f"  - {r}")


───────────────────────────────────────────────────────
Phase A: biased → neutral
───────────────────────────────────────────────────────
  Generation : 2001 total | ACCEPT 1406 (70.3%) | SKIP  595
  Audit      : 1402 total | ACCEPT 1131 (80.7%) | REJECT  271
  Overall pipeline acceptance rate : 56.7%

───────────────────────────────────────────────────────
Phase B: neutral → biased
───────────────────────────────────────────────────────
  Generation : 2317 total | ACCEPT 1058 (45.7%) | SKIP 1259
  Audit      : 1059 total | ACCEPT  697 (65.8%) | REJECT  362
  Overall pipeline acceptance rate : 30.1%

Skip reasons — Phase A:
    1x  Removing the bias removes the entire semantic point of the sentence, making a ne
    1x  The bias is deeply embedded in the structure of the conditional statement regard
    1x  The core premise is an intrusive assumption about the listener's identity; neutr
    1x  The core premise is inherently biased and difficult to neutralize without comple
    1x  The 

## 7. Build flat dataset and export

Each accepted pair yields **two rows** — the original and its counterfactual.  
They share the same `pair_id`, which is used to prevent leakage in the train/test split.

In [ ]:
def build_flat_records(accepted_pairs):
    """Convert accepted pairs into a flat list of sentence rows."""
    records = []
    for pair in accepted_pairs:
        gen    = pair["generation"]
        pair_id     = gen["pair_id"]
        source      = gen.get("source", "")
        topic       = gen.get("topic", "general")
        hasbias_orig = gen.get("hasbias_original", True)

        # Original sentence
        records.append({
            "pair_id":     pair_id,
            "sentence_id": "orig_" + str(gen["id"]),
            "original_id": gen["id"],
            "text":        gen.get("original_text", ""),
            "hasbias":     hasbias_orig,
            "source":      source,
            "topic":       topic,
            "role":        "original",
            "edit_spans":  "",
        })

        # Counterfactual sentence (label is flipped)
        records.append({
            "pair_id":     pair_id,
            "sentence_id": "cf_" + str(gen["id"]),
            "original_id": gen["id"],
            "text":        gen.get("counterfactual_text", ""),
            "hasbias":     not hasbias_orig,
            "source":      source,
            "topic":       topic,
            "role":        "counterfactual",
            "edit_spans":  "|".join(gen.get("edit_spans_counterfactual") or []),
        })
    return records


flat_records = build_flat_records(all_pairs)
df_final = pd.DataFrame(flat_records)

print(f"Final dataset: {len(df_final)} sentences ({len(all_pairs)} pairs)")
print(f"  hasbias=True  : {df_final['hasbias'].sum()}")
print(f"  hasbias=False : {(~df_final['hasbias']).sum()}")
print(f"  Sources       : {df_final['source'].value_counts().to_dict()}")
print()
print("Topic distribution:")
print(df_final[df_final['role']=='original']['topic'].value_counts().to_string())
df_final.head(6)


Final dataset: 3654 sentences (1827 pairs)
  hasbias=True  : 1827
  hasbias=False : 1827
  Sources       : {'biased_corpus_only': 2262, 'gemini_only': 1392}

Topic distribution:
topic
general          696
age              167
gender           167
socioeconomic    163
political        121
nationality       99
educational       90
racial            72
sexuality         70
disability        69
religious         67
physical          46


,pair_id,sentence_id,original_id,text,hasbias,source,topic,role,edit_spans
0,cf_neutral_1444,orig_1444,1444,Avoid going out after dark in Australia – thos...,True,biased_corpus_only,nationality,original,
1,cf_neutral_1444,cf_1444,1444,Avoid going out after dark in some areas of Au...,False,biased_corpus_only,nationality,counterfactual,in some areas of
2,cf_neutral_161,orig_161,161,There isn’t a challenge too great for a devote...,True,biased_corpus_only,religious,original,
3,cf_neutral_161,cf_161,161,There are many challenges that a devoted Buddh...,False,biased_corpus_only,religious,counterfactual,There are many challenges that a devoted Buddh...
4,cf_neutral_4309,orig_4309,4309,Everyone agrees that progressives are the true...,True,biased_corpus_only,political,original,
5,cf_neutral_4309,cf_4309,4309,Progressives often advocate for equality.,False,biased_corpus_only,political,counterfactual,|often advocate for


In [ ]:
def build_flat_records(accepted_pairs):
    """Convert accepted pairs into a flat list of sentence rows (original + counterfactual)."""
    records = []
    paired_orig_ids = set()
    for pair in accepted_pairs:
        gen    = pair["generation"]
        pair_id     = gen["pair_id"]
        source      = gen.get("source", "")
        topic       = gen.get("topic", "general")
        hasbias_orig = gen.get("hasbias_original", True)
        orig_id     = gen["id"]

        paired_orig_ids.add(orig_id)
        paired_orig_ids.add(str(orig_id))

        # Original sentence
        records.append({
            "pair_id":     pair_id,
            "sentence_id": "orig_" + str(orig_id),
            "original_id": orig_id,
            "text":        gen.get("original_text", ""),
            "hasbias":     hasbias_orig,
            "source":      source,
            "topic":       topic,
            "role":        "original",
            "edit_spans":  "",
        })

        # Counterfactual sentence (label is flipped)
        records.append({
            "pair_id":     pair_id,
            "sentence_id": "cf_" + str(orig_id),
            "original_id": orig_id,
            "text":        gen.get("counterfactual_text", ""),
            "hasbias":     not hasbias_orig,
            "source":      source,
            "topic":       topic,
            "role":        "counterfactual",
            "edit_spans":  "|".join(gen.get("edit_spans_counterfactual") or []),
        })
    return records, paired_orig_ids


def build_unpaired_records(all_entries, paired_ids, source_name, hasbias_value):
    """Build rows for original entries that have NO counterfactual (not sampled or rejected)."""
    records = []
    for entry in all_entries:
        eid = entry["id"]
        if eid in paired_ids or str(eid) in paired_ids:
            continue
        records.append({
            "pair_id":     None,
            "sentence_id": "solo_" + str(eid),
            "original_id": eid,
            "text":        entry["text"],
            "hasbias":     hasbias_value,
            "source":      source_name,
            "topic":       entry.get("bias_type") or entry.get("topic") or "general",
            "role":        "unpaired",
            "edit_spans":  "",
        })
    return records


# ── Build paired records (original + counterfactual) ──────────────────────────
records_paired_biased,  paired_ids_biased  = build_flat_records(pairs_a)
records_paired_neutral, paired_ids_neutral = build_flat_records(pairs_b)

# ── Sanity check: paired IDs vs dataset IDs ───────────────────────────────────
all_biased_ids  = {e["id"] for e in biased_data}  | {str(e["id"]) for e in biased_data}
all_neutral_ids = {e["id"] for e in neutral_data} | {str(e["id"]) for e in neutral_data}
ghost_b = paired_ids_biased  - all_biased_ids
ghost_n = paired_ids_neutral - all_neutral_ids
if ghost_b:
    print(f"  WARNING: {len(ghost_b)//2} paired IDs from Phase A not found in biased_corpus_only")
if ghost_n:
    print(f"  WARNING: {len(ghost_n)//2} paired IDs from Phase B not found in gemini_only")

# ── Build unpaired records (originals without counterfactual) ─────────────────
records_unpaired_biased  = build_unpaired_records(
    biased_data, paired_ids_biased, "biased_corpus_only", True
)
records_unpaired_neutral = build_unpaired_records(
    neutral_data, paired_ids_neutral, "gemini_only", False
)

# ── Combine into final DataFrames ────────────────────────────────────────────
df_biased  = pd.DataFrame(records_paired_biased + records_unpaired_biased)
df_neutral = pd.DataFrame(records_paired_neutral + records_unpaired_neutral)

# ── Summary ───────────────────────────────────────────────────────────────────
n_paired_b   = len([r for r in records_paired_biased if r["role"] == "original"])
n_cf_b       = len([r for r in records_paired_biased if r["role"] == "counterfactual"])
n_unpaired_b = len(records_unpaired_biased)

n_paired_n   = len([r for r in records_paired_neutral if r["role"] == "original"])
n_cf_n       = len([r for r in records_paired_neutral if r["role"] == "counterfactual"])
n_unpaired_n = len(records_unpaired_neutral)

print("=" * 60)
print("biased_corpus_v2")
print("=" * 60)
print(f"  Total sentences    : {len(df_biased)}")
print(f"  Paired originals   : {n_paired_b}")
print(f"  Counterfact. (NEW) : {n_cf_b}")
print(f"  Unpaired originals : {n_unpaired_b}  (no counterfactual — kept as-is)")
print(f"  All originals      : {n_paired_b + n_unpaired_b}  (should = {len(biased_data)})")
print()
print("=" * 60)
print("gemini_only_v2")
print("=" * 60)
print(f"  Total sentences    : {len(df_neutral)}")
print(f"  Paired originals   : {n_paired_n}")
print(f"  Counterfact. (NEW) : {n_cf_n}")
print(f"  Unpaired originals : {n_unpaired_n}  (no counterfactual — kept as-is)")
print(f"  All originals      : {n_paired_n + n_unpaired_n}  (should = {len(neutral_data)})")
print()
print("=" * 60)
print("TOTALS")
print("=" * 60)
n_total = len(df_biased) + len(df_neutral)
n_old   = n_paired_b + n_unpaired_b + n_paired_n + n_unpaired_n
n_new   = n_cf_b + n_cf_n
print(f"  Total sentences    : {n_total}")
print(f"  Old (all originals): {n_old}  (should = {len(biased_data) + len(neutral_data)})")
print(f"  New (generated)    : {n_new}")
print(f"  Pairs              : {len(pairs_a) + len(pairs_b)}")

for label, df in [("biased_corpus_v2", df_biased), ("gemini_only_v2", df_neutral)]:
    if len(df):
        print(f"\n{label}:")
        print(f"  hasbias=True  : {df['hasbias'].sum()}")
        print(f"  hasbias=False : {(~df['hasbias']).sum()}")
        print(f"  role distribution: {df['role'].value_counts().to_dict()}")


biased_corpus_v2
  Total sentences    : 3608
  Paired originals   : 1131
  Counterfact. (NEW) : 1131
  Unpaired originals : 1346  (no counterfactual — kept as-is)
  All originals      : 2477  (should = 2476)

gemini_only_v2
  Total sentences    : 3013
  Paired originals   : 696
  Counterfact. (NEW) : 696
  Unpaired originals : 1621  (no counterfactual — kept as-is)
  All originals      : 2317  (should = 2317)

TOTALS
  Total sentences    : 6621
  Old (all originals): 4794  (should = 4793)
  New (generated)    : 1827
  Pairs              : 1827

biased_corpus_v2:
  hasbias=True  : 2477
  hasbias=False : 1131
  role distribution: {'unpaired': 1346, 'original': 1131, 'counterfactual': 1131}

gemini_only_v2:
  hasbias=True  : 696
  hasbias=False : 2317
  role distribution: {'unpaired': 1621, 'original': 696, 'counterfactual': 696}


In [ ]:
# ── Save biased_corpus_v2 ──────────────────────────────────────────────────────
biased_jsonl = OUT_DIR / "biased_corpus_v2.jsonl"
biased_csv   = OUT_DIR / "biased_corpus_v2.csv"

with open(biased_jsonl, "w", encoding="utf-8") as f:
    for pair in pairs_a:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")
df_biased.to_csv(biased_csv, index=False)
print(f"Saved {biased_jsonl}  ({len(pairs_a)} pairs)")
print(f"Saved {biased_csv}  ({len(df_biased)} rows)")

# ── Save gemini_only_v2 ──────────────────────────────────────────────────────
neutral_jsonl = OUT_DIR / "gemini_only_v2.jsonl"
neutral_csv   = OUT_DIR / "gemini_only_v2.csv"

with open(neutral_jsonl, "w", encoding="utf-8") as f:
    for pair in pairs_b:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")
df_neutral.to_csv(neutral_csv, index=False)
print(f"Saved {neutral_jsonl}  ({len(pairs_b)} pairs)")
print(f"Saved {neutral_csv}  ({len(df_neutral)} rows)")

print()
print("Done. Output files:")
for p in sorted(OUT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<45} {size_kb:6.1f} KB")

Saved /content/drive/MyDrive/Dataset Bias/output/biased_corpus_v2.jsonl  (1131 pairs)
Saved /content/drive/MyDrive/Dataset Bias/output/biased_corpus_v2.csv  (3608 rows)
Saved /content/drive/MyDrive/Dataset Bias/output/gemini_only_v2.jsonl  (696 pairs)
Saved /content/drive/MyDrive/Dataset Bias/output/gemini_only_v2.csv  (3013 rows)

Done. Output files:
  audit_biased_neutral.jsonl                     355.9 KB
  audit_neutral_biased.jsonl                     271.9 KB
  bias_type_classifications.jsonl                  3.5 KB
  biased_corpus_v2.csv                           584.6 KB
  biased_corpus_v2.jsonl                        1062.4 KB
  final_counterfactuals.csv                      559.5 KB
  final_counterfactuals.jsonl                   1594.1 KB
  gemini_only_v2.csv                             342.9 KB
  gemini_only_v2.jsonl                           628.8 KB
  generation_biased_neutral.jsonl               1329.7 KB
  generation_neutral_biased.jsonl               1331.0 KB
  topic_